# Kasus 07 · Irigasi pintar IoT — Tahap 2: Model Final dan Simpan Artifact

Lanjutan dari `01_pencarian_model.ipynb`. Model terpilih di tahap 1: **HistGradientBoosting**.

Isi notebook ini:
1. Pembersihan data memakai `src/features.py` (fungsi yang sama dipakai API/app).
2. Tuning hyperparameter dengan validasi berbasis waktu (4 periode).
3. Memilih threshold berdasarkan trade-off bisnis.
4. Latih ulang di train + validation, lalu **buka test set sekali** (15–31 Oktober).
5. Simpan `artifacts/model.joblib` + `artifacts/metadata.json`.
6. Cek: model bisa dimuat ulang dan memberi prediksi + alasan.

In [ ]:
import sys, json, itertools, platform, warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import sklearn

from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             confusion_matrix, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
RANDOM_STATE = 42

# Cari root project (folder yang berisi src/features.py), supaya notebook bisa dijalankan
# dari root project maupun dari folder notebooks/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "features.py").exists())
sys.path.insert(0, str(ROOT))
from src.features import (KOLOM_INPUT, FiturIrigasi, bersihkan_data_mentah, label_latih, buat_alasan)

print("Root project:", ROOT)
print("scikit-learn:", sklearn.__version__)

## 1. Load dan bersihkan data

In [ ]:
df_raw = pd.read_csv(ROOT / "data" / "irigasi.csv")
df, laporan = bersihkan_data_mentah(df_raw)

print(f"Data mentah: {len(df_raw)} baris → bersih: {len(df)} baris")
pd.Series(laporan, name="jumlah baris").to_frame()

Tabel di atas bisa langsung dipakai untuk **tabel masalah data** di README dan laporan.
Selisih jumlah baris: 150 duplikat + 8 baris jam pertama tiap petak (tidak punya pembacaan jam sebelumnya).

In [ ]:
X = df[KOLOM_INPUT]
y = df["perlu_siram"]          # label asli (untuk evaluasi)
y_latih = label_latih(df)      # label untuk training (positif malam hari → 0)

m_trainval = df["waktu"] < "2025-10-15"
m_test     = df["waktu"] >= "2025-10-15"
print(f"Train + validation: {m_trainval.sum()} baris | Test: {m_test.sum()} baris (belum dibuka)")
X.head()

## 2. Tuning hyperparameter

Setiap kombinasi dinilai di 4 periode waktu: latih di semua data **sebelum** periode, uji di periode itu.
Periode terakhir = validation (1–14 Oktober). Test set tidak disentuh.

Batas aman per tanaman ikut dihitung ulang di setiap periode (di dalam `FiturIrigasi.fit`), jadi tidak ada
informasi dari periode uji yang bocor ke pelatihan.

In [ ]:
PERIODE = [("2025-08-25", "2025-09-08"), ("2025-09-08", "2025-09-22"),
           ("2025-09-22", "2025-10-01"), ("2025-10-01", "2025-10-15")]

def buat_pipeline(**params):
    return Pipeline([
        ("fitur", FiturIrigasi()),
        ("model", HistGradientBoostingClassifier(class_weight="balanced", max_iter=300,
                                                 random_state=RANDOM_STATE, **params)),
    ])

def nilai_periode(params):
    skor, prediksi = [], []
    for awal, akhir in PERIODE:
        m_tr = df["waktu"] < awal
        m_uji = (df["waktu"] >= awal) & (df["waktu"] < akhir)
        pipe = buat_pipeline(**params).fit(X[m_tr], y_latih[m_tr])
        p = pipe.predict_proba(X[m_uji])[:, 1]
        skor.append(f1_score(y[m_uji], p >= 0.5))
        prediksi.append(pd.DataFrame({"y": y[m_uji].values, "proba": p, "periode": awal}))
    return skor, pd.concat(prediksi, ignore_index=True)

GRID = {"learning_rate": [0.03, 0.05, 0.1],
        "max_leaf_nodes": [15, 31],
        "min_samples_leaf": [20, 50]}

hasil_tuning = []
for nilai in itertools.product(*GRID.values()):
    params = dict(zip(GRID, nilai))
    skor, _ = nilai_periode(params)
    hasil_tuning.append({**params, **{f"F1 periode {i+1}": s for i, s in enumerate(skor)},
                         "F1 rata-rata": np.mean(skor), "F1 std": np.std(skor)})

tabel_tuning = pd.DataFrame(hasil_tuning).sort_values("F1 rata-rata", ascending=False).round(3)
tabel_tuning.head(8)

In [ ]:
PARAMS_TERBAIK = {k: tabel_tuning.iloc[0][k] for k in GRID}
PARAMS_TERBAIK = {k: (int(v) if float(v).is_integer() else float(v)) for k, v in PARAMS_TERBAIK.items()}
print("Parameter terbaik:", PARAMS_TERBAIK)
print(f"Selisih terbaik vs terburuk: {tabel_tuning['F1 rata-rata'].max() - tabel_tuning['F1 rata-rata'].min():.3f}")

Selisih antar kombinasi kecil. Artinya model **tidak sensitif** terhadap hyperparameter, dan yang menentukan
performa adalah fiturnya (terutama selisih terhadap batas aman tanaman, jam siang, dan prakiraan hujan).

## 3. Memilih threshold

Prediksi dari keempat periode digabung (lebih banyak baris positif daripada validation saja, jadi lebih stabil),
lalu dilihat trade-off-nya:

- **False negative (FN)** = petak perlu disiram tapi tidak ada notifikasi → **tanaman stres**.
- **False positive (FP)** = notifikasi padahal tidak perlu → kalau langsung disiram, **air terbuang dan akar bisa busuk**.

In [ ]:
_, prediksi_oof = nilai_periode(PARAMS_TERBAIK)

baris = []
for th in np.round(np.arange(0.20, 0.96, 0.05), 2):
    pred = prediksi_oof["proba"] >= th
    yy = prediksi_oof["y"]
    baris.append({"threshold": th, "F1": f1_score(yy, pred), "Precision": precision_score(yy, pred),
                  "Recall": recall_score(yy, pred),
                  "FP (siram percuma)": int((pred & (yy == 0)).sum()),
                  "FN (telat siram)": int((~pred & (yy == 1)).sum())})
tabel_threshold = pd.DataFrame(baris).set_index("threshold").round(3)

fig, ax = plt.subplots(figsize=(8, 4))
tabel_threshold[["F1", "Precision", "Recall"]].plot(ax=ax, marker=".")
ax.set_title("Trade-off threshold (gabungan 4 periode validasi)")
ax.set_xlabel("threshold")
ax.grid(alpha=0.3)
plt.show()
tabel_threshold

F1 hampir rata di rentang threshold yang lebar. Aturan pemilihan:

> Pilih **threshold paling rendah yang F1-nya masih dalam 0,005 dari F1 maksimum.**

Alasan bisnis: kalau F1-nya praktis sama, lebih baik condong ke **recall** (sedikit telat siram), karena
agronom tidak standby 24 jam. Notifikasi yang terlewat berarti tanaman stres tanpa ada yang tahu.
Sebaliknya, notifikasi palsu masih bisa dicek dulu oleh pengelola sebelum benar-benar menyiram.

In [ ]:
f1_maks = tabel_threshold["F1"].max()
THRESHOLD = float(tabel_threshold.index[tabel_threshold["F1"] >= f1_maks - 0.005].min())
print(f"F1 maksimum: {f1_maks:.3f} | threshold dipilih: {THRESHOLD}")
tabel_threshold.loc[[THRESHOLD]]

## 4. Latih model final di train + validation

Setelah parameter dan threshold dipilih, model dilatih ulang memakai semua data sebelum 15 Oktober.

In [ ]:
model_final = buat_pipeline(**PARAMS_TERBAIK).fit(X[m_trainval], y_latih[m_trainval])
BATAS_AMAN = model_final.named_steps["fitur"].batas_aman_
print("Batas aman per tanaman (dari train + validation):", BATAS_AMAN)

## 5. Evaluasi di test set (dibuka sekali)

Dibandingkan dengan baseline wajib (`kelembaban_tanah_pct` mentah < 35, persis seperti brief) dan aturan domain.

In [ ]:
X_test, y_test = X[m_test], y[m_test]
proba_test = model_final.predict_proba(X_test)[:, 1]
pred_model = (proba_test >= THRESHOLD).astype(int)

# Baseline wajib: kelembaban tanah MENTAH (tanpa pembersihan) di jam itu < 35
mentah = df_raw.drop_duplicates(subset=["waktu", "petak"]).assign(waktu=lambda d: pd.to_datetime(d["waktu"]))
mentah = mentah.set_index(["waktu", "petak"])["kelembaban_tanah_pct"]
tanah_mentah_test = mentah.reindex(pd.MultiIndex.from_frame(df.loc[m_test, ["waktu", "petak"]])).values
pred_wajib = (tanah_mentah_test < 35).astype(int)

# Aturan domain: di bawah batas aman tanaman, siang, dan prakiraan hujan <= 2 mm
pred_domain = ((X_test["kelembaban_tanah_pct"] < X_test["tanaman"].map(BATAS_AMAN))
               & X_test["jam"].between(5, 17) & (X_test["prakiraan_hujan_6jam_mm"] <= 2)).astype(int)

def metrik(y_true, y_pred):
    return {"F1": f1_score(y_true, y_pred), "Precision": precision_score(y_true, y_pred, zero_division=0),
            "Recall": recall_score(y_true, y_pred),
            "FP (siram percuma)": int(((y_pred == 1) & (y_true == 0)).sum()),
            "FN (telat siram)": int(((y_pred == 0) & (y_true == 1)).sum())}

hasil_test = pd.DataFrame({
    "Baseline wajib (< 35)": metrik(y_test, pred_wajib),
    "Aturan domain": metrik(y_test, pred_domain),
    f"HistGradientBoosting (thr {THRESHOLD})": metrik(y_test, pred_model),
}).T.round(3)
print(f"Test: {len(y_test)} baris, {int(y_test.sum())} positif")
hasil_test

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred_model),
                       display_labels=["tidak siram", "siram"]).plot(ax=axes[0], colorbar=False)
axes[0].set_title(f"HistGradientBoosting — test (thr {THRESHOLD})")

per_tanaman = (pd.DataFrame({"tanaman": X_test["tanaman"].values, "y": y_test.values, "pred": pred_model})
               .groupby("tanaman").apply(lambda g: pd.Series({
                   "F1": f1_score(g["y"], g["pred"]), "positif": int(g["y"].sum())})))
per_tanaman["F1"].plot.bar(ax=axes[1], rot=0)
axes[1].set_title("F1 test per tanaman")
axes[1].set_ylim(0, 1)
plt.tight_layout()
plt.show()
per_tanaman.round(3)

### Kesalahan model di test
Lihat baris yang salah prediksi, untuk bahan bagian keterbatasan di laporan.

In [ ]:
salah = df.loc[m_test, ["waktu", "petak"] + KOLOM_INPUT + ["perlu_siram"]].copy()
salah["proba"] = proba_test.round(3)
salah["pred"] = pred_model
salah = salah[salah["pred"] != salah["perlu_siram"]]
salah["jenis"] = np.where(salah["pred"] == 1, "FP (siram percuma)", "FN (telat siram)")
print(salah["jenis"].value_counts().to_dict())
salah.drop(columns=["umur_tanaman_hari", "kelembaban_udara_pct", "cahaya_lux"]).head(15)

## 6. Simpan model dan metadata

In [ ]:
ART = ROOT / "artifacts"
ART.mkdir(exist_ok=True)
joblib.dump(model_final, ART / "model.joblib")

baris_model = hasil_test.iloc[-1]
metadata = {
    "versi_model": "1.0.0",
    "dibuat": datetime.now().isoformat(timespec="seconds"),
    "kasus": "07 - Irigasi pintar IoT",
    "algoritma": "HistGradientBoostingClassifier (scikit-learn) + FiturIrigasi",
    "hyperparameter": {**PARAMS_TERBAIK, "max_iter": 300, "class_weight": "balanced"},
    "threshold": THRESHOLD,
    "alasan_threshold": "Threshold terendah dengan F1 dalam 0,005 dari maksimum (validasi 4 periode), "
                        "condong ke recall karena notifikasi terlewat = tanaman stres.",
    "batas_aman_kelembaban_pct": BATAS_AMAN,
    "jam_siram": [5, 17],
    "fitur_input": {
        "tanaman": {"tipe": "kategori", "nilai": ["cabai", "tomat", "selada"]},
        "jenis_tanah": {"tipe": "kategori", "nilai": ["pasir", "lempung", "liat"]},
        "umur_tanaman_hari": {"tipe": "angka", "min": 0, "max": 365},
        "kelembaban_tanah_pct": {"tipe": "angka", "min": 0.1, "max": 100,
                                 "catatan": "pembacaan sensor terakhir, sebelum disiram"},
        "suhu_udara_C": {"tipe": "angka", "min": 0, "max": 50},
        "kelembaban_udara_pct": {"tipe": "angka", "min": 0, "max": 100},
        "cahaya_lux": {"tipe": "angka", "min": 0, "max": 150000},
        "curah_hujan_mm": {"tipe": "angka", "min": 0, "max": 100},
        "prakiraan_hujan_6jam_mm": {"tipe": "angka", "min": 0, "max": 200},
        "jam": {"tipe": "angka", "min": 0, "max": 23},
    },
    "urutan_kolom_input": KOLOM_INPUT,
    "data_latih": {"periode": "2025-08-01 s/d 2025-10-14", "baris": int(m_trainval.sum())},
    "metrik": {
        "validasi_4_periode_f1_rata2_thr0.5": float(tabel_tuning.iloc[0]["F1 rata-rata"]),
        "validasi_gabungan_di_threshold": tabel_threshold.loc[THRESHOLD, ["F1", "Precision", "Recall"]].astype(float).to_dict(),
        "test": {"periode": "2025-10-15 s/d 2025-10-31",
                 "f1": float(baris_model["F1"]), "precision": float(baris_model["Precision"]),
                 "recall": float(baris_model["Recall"]),
                 "f1_baseline_wajib": float(hasil_test.iloc[0]["F1"]),
                 "f1_aturan_domain": float(hasil_test.iloc[1]["F1"])},
    },
    "masalah_data": laporan,
    "versi_library": {"python": platform.python_version(), "scikit-learn": sklearn.__version__,
                      "pandas": pd.__version__, "numpy": np.__version__, "joblib": joblib.__version__},
}
with open(ART / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("Tersimpan:", [p.name for p in ART.iterdir()])

## 7. Cek: muat ulang model dan coba prediksi

Simulasi cara API/app memakai model: muat `model.joblib` + `metadata.json`, lalu prediksi satu input.

In [ ]:
model = joblib.load(ART / "model.joblib")
meta = json.load(open(ART / "metadata.json", encoding="utf-8"))

def prediksi(input_dict):
    proba = float(model.predict_proba(pd.DataFrame([input_dict]))[0, 1])
    perlu = proba >= meta["threshold"]
    return {"perlu_siram": bool(perlu), "probabilitas": round(proba, 3),
            "alasan": buat_alasan(input_dict, meta["batas_aman_kelembaban_pct"], proba, meta["threshold"])}

contoh = {
    "selada kering, siang, tidak hujan": dict(tanaman="selada", jenis_tanah="pasir", umur_tanaman_hari=30,
        kelembaban_tanah_pct=38, suhu_udara_C=24, kelembaban_udara_pct=70, cahaya_lux=40000,
        curah_hujan_mm=0, prakiraan_hujan_6jam_mm=0, jam=11),
    "selada kering, tapi mau hujan": dict(tanaman="selada", jenis_tanah="pasir", umur_tanaman_hari=30,
        kelembaban_tanah_pct=38, suhu_udara_C=22, kelembaban_udara_pct=90, cahaya_lux=15000,
        curah_hujan_mm=0, prakiraan_hujan_6jam_mm=8, jam=11),
    "cabai 38% (masih aman untuk cabai)": dict(tanaman="Cabe ", jenis_tanah="lempung", umur_tanaman_hari=50,
        kelembaban_tanah_pct=38, suhu_udara_C=25, kelembaban_udara_pct=65, cahaya_lux=50000,
        curah_hujan_mm=0, prakiraan_hujan_6jam_mm=0, jam=13),
    "tomat kering, tengah malam": dict(tanaman="tomat", jenis_tanah="liat", umur_tanaman_hari=60,
        kelembaban_tanah_pct=30, suhu_udara_C=17, kelembaban_udara_pct=95, cahaya_lux=0,
        curah_hujan_mm=0, prakiraan_hujan_6jam_mm=0, jam=1),
}
for nama, inp in contoh.items():
    h = prediksi(inp)
    print(f"■ {nama}\n  perlu_siram={h['perlu_siram']} (p={h['probabilitas']})")
    for a in h["alasan"]:
        print("   -", a)

## 8. Ringkasan

- Model final: **HistGradientBoosting** di dalam pipeline (`FiturIrigasi` → model), disimpan di
  `artifacts/model.joblib`. Pipeline langsung menerima input mentah dari app (10 kolom), termasuk nama tanaman
  yang penulisannya tidak rapi.
- `artifacts/metadata.json` berisi versi model, fitur input beserta rentang validnya (untuk validasi di API),
  threshold, batas aman tanaman, metrik, dan versi library.
- ⚠️ **Versi scikit-learn saat memuat model harus sama** dengan saat menyimpan (lihat `versi_library` di
  metadata). Tulis versi ini di `requirements.txt`.

**Langkah berikutnya:** API (FastAPI) yang memuat model + validasi input memakai rentang di metadata,
lalu UI Streamlit.